In [14]:
import pandas as pd
import requests
import zipfile
import io

cols_needed = [
    'as_of_year',
    'action_taken_name',
    'loan_purpose_name',
    'loan_type_name',
    'state_abbr',
]

keep_purpose = ['Home purchase', 'Refinancing']
keep_action  = ['Loan originated',
                'Application denied by financial institution',
                'Application approved but not accepted']

years = [2007, 2008, 2009, 2010, 2011, 2013, 2014, 2015, 2016, 2017]
all_agg = []

for year in years:
    print(f"Processing {year}...")

    url = f"https://files.consumerfinance.gov/hmda-historic-loan-data/hmda_{year}_nationwide_all-records_labels.zip"  # ← only change

    response = requests.get(url, stream=True, timeout=300)
    z = zipfile.ZipFile(io.BytesIO(response.content))
    fname = z.namelist()[0]

    chunks = pd.read_csv(
        z.open(fname),
        usecols=cols_needed,
        dtype=str,
        low_memory=False,
        chunksize=100_000
    )

    year_df = pd.concat([
        chunk[
            chunk['loan_purpose_name'].isin(keep_purpose) &
            chunk['action_taken_name'].isin(keep_action)
        ]
        for chunk in chunks
    ])

    agg = year_df.groupby([
        'as_of_year', 'state_abbr',
        'loan_purpose_name', 'loan_type_name',
        'action_taken_name'
    ]).size().reset_index(name='count')

    all_agg.append(agg)

    del year_df, chunks, z, response

    print(f"  ✓ {year} done — {len(agg)} aggregated rows")

final = pd.concat(all_agg, ignore_index=True)
final.to_csv('hmda_aggregated.csv', index=False)
print(f"Done. Final shape: {final.shape}")

Processing 2007...
  ✓ 2007 done — 1191 aggregated rows
Processing 2008...
  ✓ 2008 done — 1198 aggregated rows
Processing 2009...
  ✓ 2009 done — 1225 aggregated rows
Processing 2010...
  ✓ 2010 done — 1231 aggregated rows
Processing 2011...
  ✓ 2011 done — 1237 aggregated rows
Processing 2013...
  ✓ 2013 done — 1240 aggregated rows
Processing 2014...
  ✓ 2014 done — 1235 aggregated rows
Processing 2015...
  ✓ 2015 done — 1235 aggregated rows
Processing 2016...
  ✓ 2016 done — 1238 aggregated rows
Processing 2017...
  ✓ 2017 done — 1242 aggregated rows
Done. Final shape: (12272, 6)


In [15]:
from google.colab.files import download
download('hmda_aggregated.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import requests
import zipfile
import io
import pandas as pd

url = "https://files.consumerfinance.gov/hmda-historic-loan-data/hmda_2007_nationwide_first-lien-owner-occupied-1-4-family-records_labels.zip"

response = requests.get(url, timeout=300)
z = zipfile.ZipFile(io.BytesIO(response.content))
fname = z.namelist()[0]

df_2007 = pd.read_csv(z.open(fname), nrows=100, low_memory=False, header=0)
df_2007.head(50)[['as_of_year', 'loan_type_name', 'loan_purpose_name', 'action_taken_name', 'state_abbr']]

['as_of_year', 'respondent_id', 'agency_name', 'agency_abbr', 'agency_code', 'loan_type_name', 'loan_type', 'property_type_name', 'property_type', 'loan_purpose_name', 'loan_purpose', 'owner_occupancy_name', 'owner_occupancy', 'loan_amount_000s', 'preapproval_name', 'preapproval', 'action_taken_name', 'action_taken', 'msamd_name', 'msamd', 'state_name', 'state_abbr', 'state_code', 'county_name', 'county_code', 'census_tract_number', 'applicant_ethnicity_name', 'applicant_ethnicity', 'co_applicant_ethnicity_name', 'co_applicant_ethnicity', 'applicant_race_name_1', 'applicant_race_1', 'applicant_race_name_2', 'applicant_race_2', 'applicant_race_name_3', 'applicant_race_3', 'applicant_race_name_4', 'applicant_race_4', 'applicant_race_name_5', 'applicant_race_5', 'co_applicant_race_name_1', 'co_applicant_race_1', 'co_applicant_race_name_2', 'co_applicant_race_2', 'co_applicant_race_name_3', 'co_applicant_race_3', 'co_applicant_race_name_4', 'co_applicant_race_4', 'co_applicant_race_name_5'

,as_of_year,respondent_id,agency_name,agency_abbr,agency_code,loan_type_name,loan_type,property_type_name,property_type,loan_purpose_name,...,edit_status_name,edit_status,sequence_number,population,minority_population,hud_median_family_income,tract_to_msamd_income,number_of_owner_occupied_units,number_of_1_to_4_family_units,application_date_indicator
0,2007,3027509990,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,13361,5628.0,24.270000,45700.0,116.000000,1397.0,1828.0,0
1,2007,3027509990,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,13519,3759.0,1.360000,45700.0,99.940002,1277.0,1772.0,0
2,2007,3027509990,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Home purchase,...,NaN,NaN,13377,7430.0,13.970000,45700.0,103.320000,2207.0,2842.0,0
3,2007,3028209994,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Home purchase,...,NaN,NaN,144,7072.0,19.400000,45700.0,134.389999,2618.0,3399.0,0
4,2007,3027509990,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Home purchase,...,NaN,NaN,13360,5628.0,24.270000,45700.0,116.000000,1397.0,1828.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2007,0000057803,Federal Deposit Insurance Corporation,FDIC,3,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,Quality edit failure only,6.0,172099,4603.0,25.570000,52400.0,93.510002,1108.0,1681.0,0
96,2007,0000233031,Federal Reserve System,FRS,2,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,59845,4633.0,9.760000,52200.0,110.029999,1441.0,1801.0,0
97,2007,3027509990,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Home purchase,...,NaN,NaN,13440,7561.0,38.750000,45700.0,91.879997,2122.0,3067.0,0
98,2007,7499100008,Department of Housing and Urban Development,HUD,7,FHA-insured,2,One-to-four family dwelling (other than manufa...,1,Refinancing,...,Quality edit failure only,6.0,129813,3736.0,24.170000,63800.0,78.779999,371.0,632.0,0


,as_of_year,loan_type_name,loan_purpose_name,action_taken_name,state_abbr
0,2007,Conventional,Refinancing,Loan originated,NC
1,2007,Conventional,Refinancing,Loan originated,NC
2,2007,Conventional,Home purchase,Loan originated,NC
3,2007,Conventional,Home purchase,Loan originated,NC
4,2007,Conventional,Home purchase,Loan originated,NC
5,2007,FHA-insured,Refinancing,Loan originated,TX
6,2007,Conventional,Refinancing,Loan originated,NC
7,2007,Conventional,Home purchase,Loan originated,NaN
8,2007,Conventional,Refinancing,Loan originated,NC
9,2007,Conventional,Home purchase,Loan originated,MO


In [12]:
import requests
import zipfile
import io
import pandas as pd

url = "https://files.consumerfinance.gov/hmda-historic-loan-data/hmda_2008_nationwide_first-lien-owner-occupied-1-4-family-records_labels.zip"

response = requests.get(url, timeout=300)
z = zipfile.ZipFile(io.BytesIO(response.content))
fname = z.namelist()[0]

df_2008 = pd.read_csv(z.open(fname), nrows=100, low_memory=False, header=0)
df_2008.head(100)[['as_of_year', 'loan_type_name', 'loan_purpose_name', 'action_taken_name', 'state_abbr']]

,as_of_year,loan_type_name,loan_purpose_name,action_taken_name,state_abbr
0,2008,Conventional,Refinancing,Loan originated,SC
1,2008,Conventional,Refinancing,Loan originated,WA
2,2008,Conventional,Home purchase,Loan originated,PA
3,2008,Conventional,Home purchase,Loan originated,KS
4,2008,FHA-insured,Refinancing,Loan originated,TX
...,...,...,...,...,...
95,2008,Conventional,Refinancing,Loan originated,WA
96,2008,Conventional,Home purchase,Loan originated,WA
97,2008,Conventional,Home purchase,Loan originated,CA
98,2008,Conventional,Home purchase,Loan originated,TX


In [13]:
import requests
import zipfile
import io
import pandas as pd

years = [2007, 2008, 2009, 2010, 2011, 2013, 2014, 2015, 2016, 2017]

for year in years:
    url = f"https://files.consumerfinance.gov/hmda-historic-loan-data/hmda_{year}_nationwide_first-lien-owner-occupied-1-4-family-records_labels.zip"

    response = requests.get(url, timeout=300)
    z = zipfile.ZipFile(io.BytesIO(response.content))
    fname = z.namelist()[0]

    df = pd.read_csv(z.open(fname), usecols=['action_taken_name'], nrows=100000, low_memory=False)

    print(f"{year}: {df['action_taken_name'].unique()}")

    del df, z, response

2007: ['Loan originated']
2008: ['Loan originated']
2009: ['Loan originated']
2010: ['Loan originated']
2011: ['Loan originated']
2013: ['Loan originated']
2014: ['Loan originated']
2015: ['Loan originated']
2016: ['Loan originated']
2017: ['Loan originated']


In [16]:
import requests

url = "https://files.consumerfinance.gov/hmda-historic-loan-data/hmda_2012_nationwide_all-records_labels.zip"
response = requests.get(url, timeout=60)
print(response.status_code)  # 200 means it exists

200


In [20]:
import pandas as pd
import requests
import zipfile
import io

cols_needed = [
    'as_of_year',
    'action_taken_name',
    'loan_purpose_name',
    'loan_type_name',
    'state_abbr',
]

keep_purpose = ['Home purchase', 'Refinancing']
keep_action  = ['Loan originated',
                'Application denied by financial institution',
                'Application approved but not accepted']

years = [2012]
all_agg = []

for year in years:
    print(f"Processing {year}...")

    url = f"https://files.consumerfinance.gov/hmda-historic-loan-data/hmda_{year}_nationwide_all-records_labels.zip"  # ← only change

    response = requests.get(url, stream=True, timeout=300)
    z = zipfile.ZipFile(io.BytesIO(response.content))
    fname = z.namelist()[0]

    chunks = pd.read_csv(
        z.open(fname),
        usecols=cols_needed,
        dtype=str,
        low_memory=False,
        chunksize=100_000
    )

    year_df = pd.concat([
        chunk[
            chunk['loan_purpose_name'].isin(keep_purpose) &
            chunk['action_taken_name'].isin(keep_action)
        ]
        for chunk in chunks
    ])

    agg = year_df.groupby([
        'as_of_year', 'state_abbr',
        'loan_purpose_name', 'loan_type_name',
        'action_taken_name'
    ]).size().reset_index(name='count')

    all_agg.append(agg)

    del year_df, chunks, z, response

    print(f"  ✓ {year} done — {len(agg)} aggregated rows")

final = pd.concat(all_agg, ignore_index=True)
final.to_csv('hmda_aggregated_2012.csv', index=False)
print(f"Done. Final shape: {final.shape}")

Processing 2012...
  ✓ 2012 done — 1236 aggregated rows
Done. Final shape: (1236, 6)


In [21]:
from google.colab.files import download
download('hmda_aggregated_2012.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
import pandas as pd

df1 = pd.read_csv('hmda_aggregated.csv')        # your original file
df2 = pd.read_csv('hmda_aggregated_2012.csv')   # your 2012 file

final = pd.concat([df1, df2], ignore_index=True)
final = final.sort_values('as_of_year').reset_index(drop=True)
final.to_csv('hmda_aggregated_complete.csv', index=False)
print(final['as_of_year'].unique())  # verify all years are there

[2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017]


In [29]:

from google.colab.files import download
download('hmda_aggregated_complete.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
print(df1.shape)
print(df2.shape)
print(df1['as_of_year'].unique())
print(df2['as_of_year'].unique())

(12272, 6)
(1236, 6)
[2007 2008 2009 2010 2011 2013 2014 2015 2016 2017]
[2012]


In [30]:
import pandas as pd

df = pd.read_csv('hmda_aggregated_complete.csv')

print("=== NUMBERS FOR YOUR FUNNEL ===")

# Step 4 — your final aggregated dataset
print(f"Final aggregated rows: {len(df)}")
print(f"Total loans in dataset: {df['count'].sum():,}")

# Step 3 — breakdown by action taken
print("\nBy action taken:")
print(df.groupby('action_taken_name')['count'].sum().to_string())

# Step 2 — breakdown by loan purpose
print("\nBy loan purpose:")
print(df.groupby('loan_purpose_name')['count'].sum().to_string())

# Step 1 — by year (to show full scope)
print("\nBy year:")
print(df.groupby('as_of_year')['count'].sum().to_string())

=== NUMBERS FOR YOUR FUNNEL ===
Final aggregated rows: 13508
Total loans in dataset: 118,325,963

By action taken:
action_taken_name
Application approved but not accepted           7522234
Application denied by financial institution    27413370
Loan originated                                83390359

By loan purpose:
loan_purpose_name
Home purchase    46874275
Refinancing      71451688

By year:
as_of_year
2007    16139178
2008    10696368
2009    11819665
2010    10339473
2011     9384995
2012    12455720
2013    11261772
2014     7707831
2015     9191030
2016    10540453
2017     8789478
